In [35]:
# import modules
import datacube
import joblib
import os
import numpy as np
import xarray as xr
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.pyplot as plt
from dea_tools.classification import sklearn_flatten, sklearn_unflatten 
from dea_tools.datahandling import load_ard
from dea_tools.plotting import display_map, rgb, xr_animation
from odc.algo import mask_cleanup
from datacube.utils.cog import write_cog
# Import required packages
import math
import folium
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.patheffects as PathEffects
import matplotlib.pyplot as plt
import xarray as xr
from matplotlib import colors as mcolours
from matplotlib.animation import FuncAnimation
from pathlib import Path
from pyproj import Transformer
from shapely.geometry import box
from skimage.exposure import rescale_intensity
from tqdm.auto import tqdm
from datetime import datetime, timedelta

import odc.geo.xr
from odc.ui import image_aspect
from dea_tools.spatial import add_geobox

In [36]:
import sklearn
print(sklearn.__version__)

1.5.2


In [37]:
dc = datacube.Datacube(app='fmc')

In [38]:
# import model
model = joblib.load('/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/RF_AllBands_noLC_DEA_labeless.joblib')

In [39]:
def load_data(date, lon, lat):

    requested_date = datetime.strptime(date, '%d/%m/%Y')
    start_date = requested_date - timedelta(days=10)
    date_b = requested_date + timedelta(days=10)
    
    df = load_ard(dc=dc,
            products=['ga_s2am_ard_3', 'ga_s2bm_ard_3','ga_s2cm_ard_3'],
            measurements= ['nbart_blue','nbart_green','nbart_red','oa_fmask', 'nbart_red_edge_1' ,'nbart_red_edge_2' ,'nbart_red_edge_3','nbart_nir_1','nbart_nir_2','nbart_swir_2','nbart_swir_3', 'oa_nbart_contiguity'],
            mask_pixel_quality=False,
            x=lon,
            y=lat,
            resolution=(-20, 20),
            time = (str(start_date)[0:10], str(date_b)[0:10]),
            resampling={"*": "bilinear"},
            group_by='solar_day',
            output_crs= 'EPSG:3577')

    return df

def classify_FMC(data, model):
    """
    - data (xarray.Dataset): Sentinel-2 dataset containing required bands and optional multiple time steps. 
        
    - model (sklearn model): A pre-trained model for classification.
    
    Returns:
    - xarray.Dataset: A dataset containing the classified FMC results.
    
"""

    # Define masks. seperate cloud + shadow mask from water+ no_data mask becasue we want to do buffering of could+shadow but not water+no_data

    cloud_mask = (data.oa_fmask == 2) | (data.oa_fmask == 3)
    water_mask = (data.oa_fmask == 0) | (data.oa_nbart_contiguity == 0)

    #perfrom 1 pixel opening on cloud + shadow. three pixle dilation 
    better_cloud_mask = mask_cleanup(mask=cloud_mask, mask_filters=[("opening", 1),("dilation", 3)])

    #drop fmask from dataset before we classify
    data = data.drop(['oa_fmask', 'oa_nbart_contiguity'])

    data = data.where(data > -999, 0)

    #calculate NDVI and NDII

    data['ndii']=((data.nbart_nir_1-data.nbart_swir_2)/(data.nbart_nir_1+data.nbart_swir_2))
    data['ndvi']=((data.nbart_nir_1-data.nbart_red)/(data.nbart_nir_1+data.nbart_red))

    # change order of variables to be the same as the model expects
    data_neworder = data[['ndvi','ndii', 'nbart_blue','nbart_green','nbart_red','nbart_red_edge_1' ,'nbart_red_edge_2' ,'nbart_red_edge_3','nbart_nir_1','nbart_nir_2','nbart_swir_2','nbart_swir_3']] 
    
    #flattern the data using SKlearn_flatten
    data_flat = sklearn_flatten(data_neworder)
    
    #classify the data using the model
    print("predicting...")
    out_class = model.predict(data_flat)
    
    #return_classification to original shape
    #transpose because coords sideways when moving from Numpy to Xarray
    returned_result = sklearn_unflatten(out_class, data).transpose()
    
    #make results a dataset
    dataset_result = xr.Dataset({'LFMC':returned_result}, coords=data.coords, attrs=data.attrs)

    # print(data.attrs)
    
    #apply masks we generated before to classified data. it can be masked before we classify but then these pixels have a 0 value and it is better if it is 'no data'

    masked_data = dataset_result.where(~better_cloud_mask)
    masked_data = masked_data.where(~water_mask)
    
    #return
    return masked_data

In [40]:
def run_pixel_drill(subset, date, batch):
    # Get bounding box coordinates and add a buffer
    lon_min, lat_min, lon_max, lat_max = subset.total_bounds
    buffer = 0.04
    x = (lon_min - buffer, lon_max + buffer)
    y = (lat_min - buffer, lat_max + buffer)
    
    # try to load s2 data for this area for this day:
    try:
        sentinel_2_data = load_data(date=date, lon=x, lat=y)

        #RUN classification
        fmc = classify_FMC(sentinel_2_data, model)

    except Exception as e:
        # Handle the exception and print an error message
        print(f'Error loading data for {date}. Aborting. Error: {e}')
        # return
    #reproject validation points for data drill
    repo_day_subset = subset.to_crs('EPSG:3577')

    #create a data bucket (dictionary) to gather our data. this will be come a table
    data_bucket={}

   
    #itterate through features in shapfile
    for row in repo_day_subset.iterfeatures():
        #grab the shapfiles' attributes for our table of data
        mini_bucket = {}
        #mini bucked will be each row of the table
        mini_bucket['Site Name'] = row['properties']['Site Name']
        mini_bucket['Date'] = row['properties']['Date']
        mini_bucket['x'] = row['properties']['x']
        mini_bucket['y'] = row['properties']['y']

        point_xval, point_yval = row['geometry']['coordinates']

        for layers in sentinel_2_data.time:
            #now conduct pixel drill for our points for each time scene loaded
            dataset = fmc.LFMC.sel(time = layers)
    
            date_str = str(layers.data)[0:10]
            fmc_value = (dataset.sel(x=point_xval, y=point_yval, method="nearest")).data
    
            mini_bucket[date_str] = fmc_value
            #add to our row
        
        data_bucket[row['id']] = mini_bucket
        #add row to table

    #trun bucket into table
    output_table = pd.DataFrame.from_dict(data_bucket, orient='index')

    #change fromat of date so the file will save (remove '/')
    save_date = datetime.strptime(date, '%d/%m/%Y')
    save_date = save_date.strftime('%d-%m-%Y')

    
    #save
    output_table.to_csv(f'/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/test/FMC_estimates_200525_{save_date}_subset{batch}.csv')
    

    #add date to a txt file so we know what we have done allready

    with open('processed_dates.txt', 'a') as file:
        file.write(f'\n {date}')

## define CSV file we want to use to get ground truth points and dats

In [51]:
#Open XML file 
locations_file = '/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/FMC_estimates.csv'

locations = pd.read_csv(locations_file)

#filter out points with no coordinates
locations = locations[locations.x != 0]

#filter out points with no date or where date is nAN
locations = locations[locations.Date.notna()]

#generate a geodataframe from csv file
locations_gdf = gpd.GeoDataFrame(
    locations , geometry=gpd.points_from_xy(locations.x, locations.y), crs="EPSG:4283"
)



In [52]:
locations

,Site Name,Date,x,y,time,Satellite FMC estimate
1,Ngulla 1,4/10/2024,150.479327,-35.367073,1200,105
2,Ngulla 2,4/10/2024,150.444723,-35.360934,1200,105
3,Ngulla 3,4/10/2024,150.448402,-35.361892,1200,106
4,Ngulla 4,4/10/2024,150.453175,-35.354340,1200,122%
5,Ngulla 5,4/10/2024,150.442708,-35.367199,1200,104
...,...,...,...,...,...,...
391,cedar11,11/01/2025,150.482340,-35.205571,1200,NaN
392,cedar12,11/01/2025,150.481806,-35.205193,1200,NaN
393,cedar12,11/01/2025,150.480526,-35.204880,1200,NaN
394,cedar13,11/01/2025,150.478884,-35.203861,1200,NaN


In [46]:
#list all dates data has been collected

date_list = list(set(locations['Date'].tolist()))

# #opend list of dats allready run
# with open('processed_dates.txt', 'r') as file:
    
#     # reading the file
#     dates_done = file.read()
    
#     # replacing end splitting the text 
#     # when newline ('\n') is seen.
#     complete_dates = dates_done.split("\n")
#     print(complete_dates )
#     file.close()

# # if len(complete_dates) > 1:
    
#     # date_list = date_list complete_dates 

print(date_list)

['3/01/2025', '21/10/2024', '5/11/2024', '1/11/2024', '10/01/2025', '15/01/2025', '7/10/2024', '27/12/2024', '1/01/2024', '22/11/2024', '29/12/2024', nan, '2/01/2025', '17/11/2024', '16/11/2024', '19/11/2024', '11/10/2024', '6/01/2025', '5/10/2024', '31/12/2024', '15/04/2025', '9/01/2025', '30/10/2024', '10/10/2024', '2/02/2024', '8/01/2025', '4/10/2024', '14/10/2024', '7/01/2025', '30/12/2024', '3/10/2024', '31/10/2024', '11/01/2025', '21/11/2024', '2/10/2024', '28/10/2024', '8/10/2024', '29/10/2024', '28/12/2024', '20/11/2024', '1/01/2025', '4/01/2025', '19/10/2024', '16/01/2025', '26/12/2024', '12/10/2024']


In [43]:
#make groups:
#define batch_number for adding to output file


for date in date_list:
# date = '10/10/2024'
    batch = 1
    #an additional batch flag for use if we conduct multible pixel drills on the same date

    #filter points in geodataframe to just one date's woth
    day_subset = locations_gdf[locations_gdf['Date'] == date]
    
    # Skip if there are no points for this date
    if day_subset.empty:
        pass
        print(f'no points on {date}')
    
    #get list of all x cordinates in day subset
    x_list = list(set(day_subset['x'].tolist()))

    #define a new list 
    longitude_start_list = []

    #go through list of unique x coordinate values and extract first three values
    for x in x_list:
        short_x = int(x)
        if short_x in longitude_start_list:
            pass
        else:
            longitude_start_list.append(short_x)
            
    #put list in numerical order
    longitude_start_list = sorted(longitude_start_list)
    
    if len(longitude_start_list) > 1:

        for i in range(len(longitude_start_list)):
            if i < len(longitude_start_list) - 1: #so long as i isn't the last one in the list
                sub_group = day_subset[day_subset['x'] >= longitude_start_list[i]] #all rowns in geodataframe where x value is larger than i number in list

                sub_group = sub_group[sub_group['x'] < longitude_start_list[(i+1)]] #but smaller than the next value in list.
                run_pixel_drill(sub_group, date, batch)
                batch = batch + 1
            else:
                sub_group = day_subset[day_subset['x'] >= longitude_start_list[i]] #all rowns in geodataframe where x value is larger than last number in list

                run_pixel_drill(sub_group, date, batch)
                batch = batch + 1
    
    else:
        #conduct for all rows in list if all close enough together
        run_pixel_drill(day_subset, date, batch)
        batch = batch + 1
        
        

Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 5 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 9 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 7 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 7 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 8 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm_ard_3
    ga_s2cm_ard_3
Loading 4 time steps
predicting...
Finding datasets
    ga_s2am_ard_3
    ga_s2bm

TypeError: strptime() argument 1 must be str, not float

In [ ]:


def combine_csv_files(folder_path):
# List to hold dataframes
    df_list = []
    
    # Iterate over all files in the folder
    for file_name in os.listdir(folder_path):
        if file_name.endswith('.csv'):
            file_path = os.path.join(folder_path, file_name)
            df = pd.read_csv(file_path)
            df_list.append(df)
    
    # Combine all dataframes
    combined_df = pd.concat(df_list, ignore_index=True)
    return combined_df

# use
folder_path = '/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/generated_validation_data/test/'
combined_df = combine_csv_files(folder_path)
print(combined_df)

In [ ]:
combined_df.to_csv(f'/home/jovyan/gdata1/projects/Hazards/Fuel_moisture/test_new_validation_script.csv')